# Outliers Imputation - Magic Glasses (MG)

Dedicated notebook for MG outlier processing:
- `XXX_routine_outliers_detected.parquet` (subset of flagged outlier points)
- `XXX_routine_outliers_imputed.parquet` (routine data with outliers imputed)
- `XXX_routine_outliers_removed.parquet` (routine data with outliers removed)

Outlier flags are computed with:
- `OUTLIER_MAGIC_GLASSES_PARTIAL` (MAD15 -> MAD10)
- `OUTLIER_MAGIC_GLASSES_COMPLETE` (MAD15 -> MAD10 -> seasonal5 -> seasonal3)

In [ ]:
# Parameters with safe defaults
if (!exists("ROOT_PATH")) ROOT_PATH <- "~/workspace"
if (!exists("CONFIG_FILE_NAME")) CONFIG_FILE_NAME <- "SNT_config.json"
if (!exists("RUN_MAGIC_GLASSES_PARTIAL")) RUN_MAGIC_GLASSES_PARTIAL <- TRUE
if (!exists("RUN_MAGIC_GLASSES_COMPLETE")) RUN_MAGIC_GLASSES_COMPLETE <- FALSE
if (!exists("DEVIATION_MAD15")) DEVIATION_MAD15 <- 15
if (!exists("DEVIATION_MAD10")) DEVIATION_MAD10 <- 10
if (!exists("DEVIATION_SEASONAL5")) DEVIATION_SEASONAL5 <- 5
if (!exists("DEVIATION_SEASONAL3")) DEVIATION_SEASONAL3 <- 3
if (!exists("SEASONAL_WORKERS")) SEASONAL_WORKERS <- 1
if (!exists("DEV_SUBSET")) DEV_SUBSET <- FALSE
if (!exists("DEV_SUBSET_ADM1_N")) DEV_SUBSET_ADM1_N <- 2

if (!RUN_MAGIC_GLASSES_PARTIAL && !RUN_MAGIC_GLASSES_COMPLETE) {
  stop("At least one MG mode must be enabled: RUN_MAGIC_GLASSES_PARTIAL or RUN_MAGIC_GLASSES_COMPLETE")
}

if (SEASONAL_WORKERS < 1) {
  stop("SEASONAL_WORKERS must be >= 1")
}

if (DEV_SUBSET_ADM1_N < 1) {
  stop("DEV_SUBSET_ADM1_N must be >= 1")
}

CODE_PATH <- file.path(ROOT_PATH, "code")
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DIR <- file.path(DATA_PATH, "dhis2", "outliers_imputation")
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
source(file.path(CODE_PATH, "snt_utils.r"))

required_packages <- c("arrow", "data.table", "jsonlite", "reticulate", "glue")
if (RUN_MAGIC_GLASSES_COMPLETE) {
  required_packages <- c(required_packages, "forecast")
}
if (RUN_MAGIC_GLASSES_COMPLETE && SEASONAL_WORKERS > 1) {
  required_packages <- c(required_packages, "future", "future.apply")
}
install_and_load(unique(required_packages))

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
openhexa <- reticulate::import("openhexa.sdk")

if (RUN_MAGIC_GLASSES_COMPLETE) {
  log_msg("[WARNING] Complete mode: seasonal detection is very computationally intensive and can take several hours to run.", "warning")
}

if (RUN_MAGIC_GLASSES_COMPLETE && SEASONAL_WORKERS > 1) {
  future::plan(future::multisession, workers = SEASONAL_WORKERS)
  log_msg(glue::glue("Using parallel seasonal detection with {SEASONAL_WORKERS} workers"))
}

config_json <- fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
fixed_cols <- c("PERIOD", "YEAR", "MONTH", "ADM1_ID", "ADM2_ID", "OU_ID")
indicators_to_keep <- names(config_json$DHIS2_DATA_DEFINITIONS$DHIS2_INDICATOR_DEFINITIONS)

dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
dhis2_routine <- get_latest_dataset_file_in_memory(dataset_name, paste0(COUNTRY_CODE, "_routine.parquet"))

cols_to_select <- intersect(c(fixed_cols, indicators_to_keep), names(dhis2_routine))
dt_routine <- as.data.table(dhis2_routine)[, ..cols_to_select]

dhis2_routine_long <- melt(
  dt_routine,
  id.vars = intersect(fixed_cols, names(dt_routine)),
  measure.vars = intersect(indicators_to_keep, names(dt_routine)),
  variable.name = "INDICATOR",
  value.name = "VALUE",
  variable.factor = FALSE
)

if (DEV_SUBSET) {
  unique_adm1 <- unique(dhis2_routine_long$ADM1_ID)
  adm1_to_keep <- unique_adm1[seq_len(min(DEV_SUBSET_ADM1_N, length(unique_adm1)))]
  dhis2_routine_long <- dhis2_routine_long[ADM1_ID %in% adm1_to_keep]
  log_msg(glue::glue("DEV_SUBSET enabled: keeping {length(adm1_to_keep)} ADM1 values"), "warning")
}

log_msg(glue::glue("Data loaded: {nrow(dhis2_routine_long)} rows, {length(unique(dhis2_routine_long$OU_ID))} facilities"))

if (RUN_MAGIC_GLASSES_COMPLETE) {
  n_groups <- uniqueN(dhis2_routine_long[, .(OU_ID, INDICATOR)])
  log_msg(glue::glue("Complete mode active: seasonal detection will run on up to {n_groups} OU_ID x INDICATOR time series."), "warning")
} else {
  log_msg("Partial mode active: seasonal detection is skipped.")
}

In [ ]:
detect_outliers_mad_custom <- function(dt, deviation) {
  flag_col <- paste0("OUTLIER_MAD", deviation)
  dt <- copy(dt)
  dt[, median_val := median(VALUE, na.rm = TRUE), by = .(YEAR, OU_ID, INDICATOR)]
  dt[, mad_val := mad(VALUE, constant = 1, na.rm = TRUE), by = .(YEAR, OU_ID, INDICATOR)]
  dt[, (flag_col) := (VALUE > (median_val + deviation * mad_val)) | (VALUE < (median_val - deviation * mad_val))]
  dt[, c("median_val", "mad_val") := NULL]
  dt
}

detect_seasonal_outliers <- function(dt, deviation, workers = 1) {
  outlier_col <- paste0("OUTLIER_SEASONAL", deviation)
  dt <- copy(dt)
  setorder(dt, OU_ID, INDICATOR, PERIOD)

  process_group <- function(sub_dt) {
    n_valid <- sum(!is.na(sub_dt$VALUE))
    if (n_valid < 2) {
      return(data.table(
        PERIOD = sub_dt$PERIOD,
        OU_ID = sub_dt$OU_ID,
        INDICATOR = sub_dt$INDICATOR,
        OUTLIER_FLAG = rep(FALSE, nrow(sub_dt))
      ))
    }

    values <- as.numeric(sub_dt$VALUE)
    ts_data <- stats::ts(values, frequency = 12)
    cleaned_ts <- tryCatch(
      forecast::tsclean(ts_data, replace.missing = TRUE),
      error = function(e) ts_data
    )
    mad_val <- mad(values, constant = 1, na.rm = TRUE)

    if (is.na(mad_val) || mad_val == 0) {
      return(data.table(
        PERIOD = sub_dt$PERIOD,
        OU_ID = sub_dt$OU_ID,
        INDICATOR = sub_dt$INDICATOR,
        OUTLIER_FLAG = rep(FALSE, nrow(sub_dt))
      ))
    }

    is_outlier <- abs(as.numeric(ts_data) - as.numeric(cleaned_ts)) / mad_val >= deviation
    is_outlier[is.na(is_outlier)] <- FALSE

    data.table(
      PERIOD = sub_dt$PERIOD,
      OU_ID = sub_dt$OU_ID,
      INDICATOR = sub_dt$INDICATOR,
      OUTLIER_FLAG = as.logical(is_outlier)
    )
  }

  group_keys <- unique(dt[, .(OU_ID, INDICATOR)])
  group_list <- lapply(seq_len(nrow(group_keys)), function(i) {
    dt[OU_ID == group_keys$OU_ID[i] & INDICATOR == group_keys$INDICATOR[i]]
  })

  if (workers > 1 && requireNamespace("future.apply", quietly = TRUE)) {
    result_list <- future.apply::future_lapply(group_list, process_group, future.seed = TRUE)
  } else {
    result_list <- lapply(group_list, process_group)
  }

  outlier_flags <- rbindlist(result_list, use.names = TRUE)
  setnames(outlier_flags, "OUTLIER_FLAG", outlier_col)

  result_dt <- merge(dt, outlier_flags, by = c("PERIOD", "OU_ID", "INDICATOR"), all.x = TRUE)
  result_dt[is.na(get(outlier_col)), (outlier_col) := FALSE]
  result_dt
}

In [ ]:
if (RUN_MAGIC_GLASSES_PARTIAL | RUN_MAGIC_GLASSES_COMPLETE) {
  log_msg("Starting MAD15 detection...")
  flagged_outliers_mad15 <- detect_outliers_mad_custom(dhis2_routine_long, DEVIATION_MAD15)
  flagged_outliers_mad15_filtered <- flagged_outliers_mad15[OUTLIER_MAD15 == FALSE]

  log_msg("Starting MAD10 detection...")
  flagged_outliers_mad10 <- detect_outliers_mad_custom(flagged_outliers_mad15_filtered, DEVIATION_MAD10)
  setnames(flagged_outliers_mad10, paste0("OUTLIER_MAD", DEVIATION_MAD10), "OUTLIER_MAD15_MAD10")

  join_cols <- c("PERIOD", "OU_ID", "INDICATOR")
  mad10_subset <- flagged_outliers_mad10[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_MAD15_MAD10)]
  flagged_outliers_mad15_mad10 <- merge(
    flagged_outliers_mad15,
    mad10_subset,
    by = join_cols,
    all.x = TRUE
  )
  flagged_outliers_mad15_mad10[is.na(OUTLIER_MAD15_MAD10), OUTLIER_MAD15_MAD10 := TRUE]
  log_msg(glue::glue("MAD partial done: {sum(flagged_outliers_mad15_mad10$OUTLIER_MAD15_MAD10)} outliers flagged"))
}

if (RUN_MAGIC_GLASSES_COMPLETE) {
  flagged_outliers_mad15_mad10_filtered <- flagged_outliers_mad15_mad10[OUTLIER_MAD15_MAD10 == FALSE]

  if (nrow(flagged_outliers_mad15_mad10_filtered) == 0) {
    log_msg("No rows left after MAD partial filtering; seasonal step will be skipped.", "warning")
    flagged_outliers_seasonal5 <- copy(flagged_outliers_mad15_mad10_filtered)
    flagged_outliers_seasonal5[, OUTLIER_SEASONAL5 := FALSE]
    flagged_outliers_seasonal5_filtered <- flagged_outliers_seasonal5
    flagged_outliers_seasonal3 <- copy(flagged_outliers_seasonal5_filtered)
    flagged_outliers_seasonal3[, OUTLIER_SEASONAL3 := FALSE]
  } else {
    log_msg(glue::glue("Starting SEASONAL5 detection on {nrow(flagged_outliers_mad15_mad10_filtered)} rows..."))
    t_seasonal5 <- system.time({
      flagged_outliers_seasonal5 <- detect_seasonal_outliers(
        flagged_outliers_mad15_mad10_filtered,
        deviation = DEVIATION_SEASONAL5,
        workers = SEASONAL_WORKERS
      )
    })
    flagged_outliers_seasonal5_filtered <- flagged_outliers_seasonal5[OUTLIER_SEASONAL5 == FALSE]
    log_msg(glue::glue("SEASONAL5 finished in {round(t_seasonal5['elapsed'], 1)}s. Remaining rows: {nrow(flagged_outliers_seasonal5_filtered)}"))

    log_msg(glue::glue("Starting SEASONAL3 detection on {nrow(flagged_outliers_seasonal5_filtered)} rows..."))
    t_seasonal3 <- system.time({
      flagged_outliers_seasonal3 <- detect_seasonal_outliers(
        flagged_outliers_seasonal5_filtered,
        deviation = DEVIATION_SEASONAL3,
        workers = SEASONAL_WORKERS
      )
    })
    log_msg(glue::glue("SEASONAL3 finished in {round(t_seasonal3['elapsed'], 1)}s."))
  }

  setnames(flagged_outliers_seasonal3, paste0("OUTLIER_SEASONAL", DEVIATION_SEASONAL3), "OUTLIER_SEASONAL5_SEASONAL3")

  seasonal3_subset <- flagged_outliers_seasonal3[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_SEASONAL5_SEASONAL3)]
  flagged_outliers_seasonal5_seasonal3 <- merge(
    flagged_outliers_seasonal5,
    seasonal3_subset,
    by = join_cols,
    all.x = TRUE
  )
  flagged_outliers_seasonal5_seasonal3[is.na(OUTLIER_SEASONAL5_SEASONAL3), OUTLIER_SEASONAL5_SEASONAL3 := TRUE]
  log_msg(glue::glue("SEASONAL complete done: {sum(flagged_outliers_seasonal5_seasonal3$OUTLIER_SEASONAL5_SEASONAL3)} outliers flagged"))
}

In [ ]:
base_cols <- intersect(c(fixed_cols, "INDICATOR", "VALUE"), names(dhis2_routine_long))
flagged_outliers_mg <- copy(dhis2_routine_long[, ..base_cols])
join_cols <- c("PERIOD", "OU_ID", "INDICATOR")

if (RUN_MAGIC_GLASSES_PARTIAL | RUN_MAGIC_GLASSES_COMPLETE) {
  partial_subset <- flagged_outliers_mad15_mad10[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_MAD15_MAD10)]
  flagged_outliers_mg <- merge(flagged_outliers_mg, partial_subset, by = join_cols, all.x = TRUE)
  setnames(flagged_outliers_mg, "OUTLIER_MAD15_MAD10", "OUTLIER_MAGIC_GLASSES_PARTIAL")
}

if (RUN_MAGIC_GLASSES_COMPLETE) {
  complete_subset <- flagged_outliers_seasonal5_seasonal3[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_SEASONAL5_SEASONAL3)]
  flagged_outliers_mg <- merge(flagged_outliers_mg, complete_subset, by = join_cols, all.x = TRUE)
  setnames(flagged_outliers_mg, "OUTLIER_SEASONAL5_SEASONAL3", "OUTLIER_MAGIC_GLASSES_COMPLETE")
  flagged_outliers_mg[is.na(OUTLIER_MAGIC_GLASSES_COMPLETE) & OUTLIER_MAGIC_GLASSES_PARTIAL == TRUE, 
                      OUTLIER_MAGIC_GLASSES_COMPLETE := TRUE]
}

if ("OUTLIER_MAGIC_GLASSES_PARTIAL" %in% names(flagged_outliers_mg)) {
  flagged_outliers_mg[is.na(OUTLIER_MAGIC_GLASSES_PARTIAL), OUTLIER_MAGIC_GLASSES_PARTIAL := FALSE]
}
if ("OUTLIER_MAGIC_GLASSES_COMPLETE" %in% names(flagged_outliers_mg)) {
  flagged_outliers_mg[is.na(OUTLIER_MAGIC_GLASSES_COMPLETE), OUTLIER_MAGIC_GLASSES_COMPLETE := FALSE]
}

active_outlier_col <- if (
  RUN_MAGIC_GLASSES_COMPLETE && "OUTLIER_MAGIC_GLASSES_COMPLETE" %in% names(flagged_outliers_mg)
) {
  "OUTLIER_MAGIC_GLASSES_COMPLETE"
} else {
  "OUTLIER_MAGIC_GLASSES_PARTIAL"
}

if (!(active_outlier_col %in% names(flagged_outliers_mg))) {
  stop(glue::glue("Expected outlier flag column not found: {active_outlier_col}"))
}

# 1) Detected outliers (subset)
detected_tbl <- flagged_outliers_mg[get(active_outlier_col) == TRUE]
arrow::write_parquet(detected_tbl, file.path(OUTPUT_DIR, paste0(COUNTRY_CODE, "_routine_outliers_detected.parquet")))
log_msg(glue::glue("Exported {nrow(detected_tbl)} detected outliers to {COUNTRY_CODE}_routine_outliers_detected.parquet"))

# Helper to restore routine dataset format (same structure as input routine table)
to_routine_wide <- function(dt_long) {
  routine_wide <- dcast(
    dt_long[, .(PERIOD, YEAR, MONTH, ADM1_ID, ADM2_ID, OU_ID, INDICATOR, VALUE)],
    PERIOD + YEAR + MONTH + ADM1_ID + ADM2_ID + OU_ID ~ INDICATOR,
    value.var = "VALUE"
  )

  missing_cols <- setdiff(names(dt_routine), names(routine_wide))
  if (length(missing_cols) > 0) {
    for (col in missing_cols) {
      routine_wide[, (col) := NA_real_]
    }
  }

  target_cols <- names(dt_routine)
  routine_wide <- routine_wide[, ..target_cols]
  routine_wide
}

# 2) Imputed routine data (same moving-average logic as other outlier pipelines)
imputed_long <- copy(flagged_outliers_mg)
setorder(imputed_long, ADM1_ID, ADM2_ID, OU_ID, INDICATOR, PERIOD, YEAR, MONTH)
imputed_long[, TO_IMPUTE := fifelse(get(active_outlier_col) == TRUE, NA_real_, VALUE)]
imputed_long[
  , MOVING_AVG := frollapply(
      TO_IMPUTE,
      n = 3,
      FUN = function(x) ceiling(mean(x, na.rm = TRUE)),
      align = "center"
    ),
  by = .(ADM1_ID, ADM2_ID, OU_ID, INDICATOR)
]
imputed_long[, VALUE_IMPUTED := fifelse(is.na(TO_IMPUTE), MOVING_AVG, TO_IMPUTE)]
imputed_long[, VALUE := VALUE_IMPUTED]
imputed_long[, c("TO_IMPUTE", "MOVING_AVG", "VALUE_IMPUTED") := NULL]

routine_imputed <- to_routine_wide(imputed_long)
arrow::write_parquet(routine_imputed, file.path(OUTPUT_DIR, paste0(COUNTRY_CODE, "_routine_outliers_imputed.parquet")))
log_msg(glue::glue("Exported routine imputed table to {COUNTRY_CODE}_routine_outliers_imputed.parquet"))

# 3) Removed routine data
removed_long <- copy(flagged_outliers_mg)
removed_long[get(active_outlier_col) == TRUE, VALUE := NA_real_]

routine_removed <- to_routine_wide(removed_long)
arrow::write_parquet(routine_removed, file.path(OUTPUT_DIR, paste0(COUNTRY_CODE, "_routine_outliers_removed.parquet")))
log_msg(glue::glue("Exported routine removed table to {COUNTRY_CODE}_routine_outliers_removed.parquet"))

log_msg("MG outlier tables exported successfully.")